ChatGroq API

In [2]:
from langchain_groq.chat_models import ChatGroq
# Groq_Token = 'gsk_OaiU........' # Never hardcode API keys in your code, especially if you plan to share it.

# We can use environment variables to for hiding the API key.
import os
Groq_Token = os.getenv('CHAT_GROQ_API_KEY')

In [ ]:
#Code block to check for available models using the Groq API (code borrowed from https://console.groq.com/docs/models)
import requests

api_key = Groq_Token
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'openai/gpt-oss-safeguard-20b', 'object': 'model', 'created': 1761708789, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'groq/compound', 'object': 'model', 'created': 1756949530, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'canopylabs/orpheus-v1-english', 'object': 'model', 'created': 1766186316, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'allam-2-7b', 'object': 'model', 'created': 1737672203, 'owned_by': 'SDAIA', 'active': True, 'context_window': 4096, 'public_apps': None, 'max_completion_tokens': 4096}, {'id': 'meta-llama/llama-prompt-guard-2-22m', 'object': 'model', 'created': 1748632101, 'owned_by': 'Meta', 'active': True, 'context_window': 512, 'public_apps': None, 'max_completion_tokens': 512}, {'id': 'meta-

In [ ]:
# Groq LLM model initialization
model_name = "llama-3.1-8b-instant"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

# Statement 
sentence = "The product quality is amazing but the delivery was delayed. However I am happy with the customer service."

### Zero Shot

We asking the model to perform a task without providing any examples. The model relies entirely on its pre-trained knowledge and the clarity of the instruction.
- Clear instructions are critical to produce better outputs
- Works well for simple or common tasks, fails on complex tasks

In [ ]:
# System Prompts 
query = f"""
* You are a sentiment analysis model. 
* Your task is to analyze the sentiment expressed in the given text and classify it as 'positive', 'negative', or 'neutral'. 
* Provide the sentiment label and, if necessary, a brief explanation of your reasoning.

Sentence: {sentence}
""" 

answer = llm.invoke(query)
print(answer.content)

Sentiment Label: Neutral

Explanation: 
The sentence contains both positive and negative sentiments. The phrase "The product quality is amazing" expresses a strong positive sentiment, indicating satisfaction with the product. However, the phrase "the delivery was delayed" expresses a negative sentiment, indicating a problem with the delivery process. The phrase "However I am happy with the customer service" further adds a positive sentiment, balancing out the negative sentiment. Overall, the sentence can be classified as neutral because the positive and negative sentiments cancel each other out, resulting in a neutral tone.


### Few Shot

We provides a few examples to guide the model. This helps the model understand the expected pattern and improves accuracy.
- Demonstrates input-output format
- Reduces ambiguity

In [ ]:
# System Prompts 
query = f"""
* You are a sentiment analysis model. 
* Your task is to analyze the sentiment expressed in the given text and classify it as 'positive', 'negative', or 'neutral'. 
* Provide the sentiment label and, if necessary, a brief explanation of your reasoning.

Here are few examples:
1. Sentence: 'The customer service was excellent, and I received my order quickly.'
Sentiment: Positive

2. Sentence: 'The food was bland and the service was slow.'
Sentiment: Negative

3. Sentence: 'The product is okay, but it's not worth the price.'
Sentiment: Neutral

Sentence: {sentence}
""" 

answer = llm.invoke(query)
print(answer.content)

Sentiment: Positive

My reasoning is as follows: The text mentions two negative aspects - 'the delivery was delayed'. However, it also mentions two positive aspects - 'the product quality is amazing' and 'I am happy with the customer service'. The positive aspects seem to outweigh the negative aspect, resulting in a positive overall sentiment.


### Role-based prompting

We assign a specific persona or role to the model. This influences tone, style, and sometimes reasoning approach.
- "You are X..." defines behavior
- Useful for creative, domain-specific, or stylistic tasks

In [20]:
model_name = "llama-3.3-70b-versatile"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

In [ ]:
# without persona
query = "Write me a short poem"
answer = llm.invoke(query)
print(answer.content)

Here's a short poem:

The sun sets slow and paints the sky,
A fiery hue that makes me sigh.
The stars come out and twinkle bright,
A night of rest, a peaceful sight.

The world is calm, the darkness deep,
A time for dreams, a time to sleep.
The moon's soft light, a gentle beam,
Guides me through, a peaceful dream.


In [ ]:
# with persona
query = "You are Shakespeare, an English writer. Write me a short poem"
answer = llm.invoke(query)
print(answer.content)

Fairest of requests, I shall comply,
And pen a verse, to catch thy gentle eye.

"Oh, moonlit night, with stars above,
Thy silvery glow, doth fill my love.
The world, in slumber, doth quietly sleep,
 Whilst I, awake, my heart doth softly keep.

The wind, a whisper, through the trees doth stray,
And in its sighs, I hear a gentle sway.
The night, a mistress, doth entice my soul,
To dance, beneath her starry, heavenly role."

Thus, I hope, this humble verse, doth please thy mind,
And in its words, a glimpse of beauty, thou dost find.


### System Prompting

Instead of writing everything in one string, we separate instructions using system and user messages.
- System message defines behavior
- User message defines the task
- More structured and reliable

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant that explains concepts in simple terms."),
    HumanMessage(content="Explain what a black hole is in 2 lines.")
]

response = llm.invoke(messages)
print(response.content)

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape once it gets too close. It's formed when a massive star collapses in on itself, creating a massive vacuum that warps the fabric of space and time around it.


### Structured Reasoning

This pattern explicitly asks the model to reason step by step before giving the answer.
- Encourages logical breakdown
- Improves accuracy on multi-step problems

In [5]:
query = """
Solve the problem step by step:

If a train travels 60 km in 1 hour, how far will it travel in 5 hours?

Show reasoning clearly.
"""

response = llm.invoke(query)
print(response.content)

To solve this problem, we need to find the distance the train will travel in 5 hours. We know the speed of the train (60 km/h) and the time it will travel (5 hours). 

**Step 1: Identify the given information**

- Speed of the train: 60 km/h
- Time the train will travel: 5 hours

**Step 2: Recall the formula for distance**

The formula for distance is: Distance = Speed × Time

**Step 3: Plug in the values**

We know the speed (60 km/h) and the time (5 hours). We will substitute these values into the formula.

Distance = 60 km/h × 5 hours

**Step 4: Calculate the distance**

To find the distance, we multiply the speed by the time.

Distance = 60 km/h × 5 hours
= 60 × 5
= 300 km

**Step 5: Write the final answer**

The train will travel 300 km in 5 hours.


### Output Format Control

In [6]:
# JSON
query = """
Extract information from the text and return ONLY valid JSON.

Text: "John bought a laptop for $1200 on 5th May 2026."

Format:
{
  "name": "",
  "item": "",
  "price": "",
  "date": ""
}
"""

response = llm.invoke(query)
print(response.content)

{
  "name": "John",
  "item": "laptop",
  "price": "1200",
  "date": "5th May 2026"
}


In [7]:
# Lists
query = """
List 5 programming languages.

Return as a numbered list only.
"""

response = llm.invoke(query)
print(response.content)

1. Python
2. Java
3. JavaScript
4. C++
5. Ruby


In [13]:
# Structured text
query = """
Summarize the topic "Linked List" in this format:

Title:
Definition:
Applications:
Advantages:
"""

response = llm.invoke(query)
print(response.content)

**Linked List**

**Definition:** 
A linked list is a linear data structure where each element is a separate object, known as a node. Each node contains two items: the data and a reference (or link) to the next node in the sequence. This structure allows for efficient insertion or removal of elements from any position in the list.

**Applications:**
1. **Database Query Results**: Linked lists are often used to store query results in databases, as they allow for efficient insertion and removal of rows.
2. **Browser History**: Web browsers use linked lists to store the history of visited web pages, allowing for easy navigation between pages.
3. **Dynamic Memory Allocation**: Linked lists are used in memory management to allocate and deallocate memory dynamically.
4. **File Systems**: Linked lists are used in file systems to manage file metadata and directory structures.

**Advantages:**
1. **Efficient Insertion and Deletion**: Linked lists allow for efficient insertion and deletion of ele

### Self-Checking prompts

The model generates an answer, then **critiques and improves its own response**.
- Introduces self-evaluation
- Often improves quality

In [10]:
def self_checking_prompt(task):
    prompt = f"""
You are an expert assistant.

Step 1: Solve the task.
Step 2: Critically review your answer for correctness, clarity, and completeness.
Step 3: Provide a refined final answer.

Task:
{task}

Return in this format:
Initial Answer:
Critique:
Final Answer:
"""
    response = llm.invoke(prompt)
    return response.content


task = "Compute factorial of 5."
print(self_checking_prompt(task))

**Step 1: Solve the task.**

The factorial of a number is the product of all positive integers less than or equal to that number. To compute the factorial of 5, we can use the following formula:

5! = 5 × 4 × 3 × 2 × 1

Using this formula, we can calculate the factorial of 5 as follows:

5! = 5 × 4 = 20
20 × 3 = 60
60 × 2 = 120
120 × 1 = 120

Therefore, the factorial of 5 is 120.

**Initial Answer:**
The factorial of 5 is 120.

**Step 2: Critically review my answer for correctness, clarity, and completeness.**

Upon reviewing my answer, I can see that it is correct. The calculation is straightforward and easy to follow. However, I can improve the clarity of my answer by providing a more detailed explanation of the formula and the calculation process.

Additionally, I can provide a more concise answer by using mathematical notation. For example, I can write 5! = 5 × 4 × 3 × 2 × 1 as 5! = 5 × 4! = 5 × (4 × 3 × 2 × 1).

**Critique:**
My answer is correct, but it can be improved by providi

### Iterative refinement

The model improves its output over multiple iterations.
- Multi-step improvement loop
- Simulates deeper thinking

In [14]:
def iterative_refinement(task, iterations=3):
    answer = ""

    for i in range(iterations):
        prompt = f"""
Improve the following answer:

Task: {task}

Current Answer:
{answer}

Provide a better version.
"""
        response = llm.invoke(prompt)
        answer = response.content

    return answer


print(iterative_refinement("Explain dynamic programming in simple terms."))

**Dynamic Programming in Simple Terms**

**What is Dynamic Programming?**

Dynamic programming is a problem-solving approach that helps you find the most efficient solution to a complex problem by breaking it down into smaller, more manageable sub-problems. It's a systematic method that allows you to solve problems by combining the solutions of smaller sub-problems, rather than solving the entire problem from scratch.

**A Real-World Analogy: Planning a Road Trip**

Imagine you're planning a road trip from New York to Los Angeles. You have a map, and you know the distance between each city. To find the shortest route, you could try every possible combination of roads, but that would take an impractically long time. Instead, you can use dynamic programming to break down the problem into smaller sub-problems:

1. Find the shortest route from New York to Chicago.
2. Find the shortest route from Chicago to Denver.
3. Find the shortest route from Denver to Los Angeles.

By solving each sub-

### Prompt Versioning

Prompts are treated like code and improved over time with **version tracking**.
- Maintain v1, v2, v3...
- Track performance changes

In [ ]:
prompt_log = [
    {
        "version": "v1",
        "prompt": "Explain AI.",
        "avg_score": 5.2,
        "notes": "Too vague"
    },
    {
        "version": "v2",
        "prompt": "Explain AI with examples.",
        "avg_score": 7.8,
        "notes": "Better clarity"
    },
    {
        "version": "v3",
        "prompt": "Explain AI as a teacher with examples, 5 lines max.",
        "avg_score": 9.1,
        "notes": "Best so far"
    }
]

### Scoring prompts using llm

In [15]:
test_inputs = [
    "Explain recursion simply.",
    "Explain recursion in one sentence.",
    "Explain recursion to a 5-year-old.",
    "Give a technical definition of recursion.",
    "Explain recursion with Python code.",
    "Explain recursion without using the word 'recursion'.",
    "Explain recursion with a real-world analogy.",
    "Why is recursion hard to understand?",
    "Compare recursion vs iteration.",
    "Give a wrong explanation of recursion and then correct it."
]

In [16]:
def run_prompt_on_dataset(prompt_template, inputs):
    results = []

    for inp in inputs:
        query = prompt_template.replace("{input}", inp)
        response = llm.invoke(query).content

        results.append({
            "input": inp,
            "output": response
        })

    return results

In [17]:
def llm_judge(input_text, output_text):
    judge_prompt = f"""
Evaluate the response based on:
- Correctness
- Clarity
- Instruction following

Score from 1 to 10.

Input:
{input_text}

Output:
{output_text}

Return only a number.
"""
    score = llm.invoke(judge_prompt).content
    return score

### Testing Harness

Prompts are tested across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases

In [18]:
def test_harness(prompt_template, inputs, use_llm_judge=False):
    results = []

    for inp in inputs:
        query = prompt_template.replace("{input}", inp)
        output = llm.invoke(query).content

        score = llm_judge(inp, output)
        results.append({
            "input": inp,
            "output": output,
            "score": score
        })

    return results

In [21]:
prompt_v1 = "Explain the following:\n{input}"

results = test_harness(prompt_v1, test_inputs)

for r in results:
    print("\n---")
    print("Input:", r["input"])
    print("Score:", r["score"])


---
Input: Explain recursion simply.
Score: 9

---
Input: Explain recursion in one sentence.
Score: 10

---
Input: Explain recursion to a 5-year-old.
Score: 9

---
Input: Give a technical definition of recursion.
Score: 9

---
Input: Explain recursion with Python code.
Score: 8

---
Input: Explain recursion without using the word 'recursion'.
Score: 9

---
Input: Explain recursion with a real-world analogy.
Score: 9

---
Input: Why is recursion hard to understand?
Score: 9

---
Input: Compare recursion vs iteration.
Score: 9

---
Input: Give a wrong explanation of recursion and then correct it.
Score: 8


### Failure testing

Identify and analyze cases where the model performs poorly.
- Focus on weaknesses, not just averages
- Drives prompt improvement

In [24]:
def get_failures(results, threshold=8.5):
    return [r for r in results if float(r["score"]) < threshold]


failures = get_failures(results)

for f in failures:
    print("\nFAILED CASE:")
    print("Input:", f["input"])
    print("Score:", f["score"])


FAILED CASE:
Input: Explain recursion with Python code.
Score: 8

FAILED CASE:
Input: Give a wrong explanation of recursion and then correct it.
Score: 8


### Evaluation and Comparision

Compare different prompt versions using **scores and outputs**.
- Evidence-based improvement
- Use metrics, not intuition

In [25]:
prompt_v2 = """
You are a teacher.

Explain clearly and concisely.
Follow the instruction strictly.

Task:
{input}
"""

results_v2 = test_harness(prompt_v2, test_inputs)

In [26]:
def average_score(results):
    return sum(float(r["score"]) for r in results) / len(results)


print("V1 Avg:", average_score(results))
print("V2 Avg:", average_score(results_v2))

V1 Avg: 8.9
V2 Avg: 8.9


In [27]:
version_log = [
    {
        "version": "v1",
        "prompt": prompt_v1,
        "avg_score": average_score(results),
        "failures": len(get_failures(results)),
        "notes": "Too generic"
    },
    {
        "version": "v2",
        "prompt": prompt_v2,
        "avg_score": average_score(results_v2),
        "failures": len(get_failures(results_v2)),
        "notes": "Better instruction following"
    }
]

for v in version_log:
    print(v)

{'version': 'v1', 'prompt': 'Explain the following:\n{input}', 'avg_score': 8.9, 'failures': 2, 'notes': 'Too generic'}
{'version': 'v2', 'prompt': '\nYou are a teacher.\n\nExplain clearly and concisely.\nFollow the instruction strictly.\n\nTask:\n{input}\n', 'avg_score': 8.9, 'failures': 1, 'notes': 'Better instruction following'}
